# Vulnerability Classification System

This project implements an advanced vulnerability classification system utilizing various Language Models (LLMs) and sophisticated prompt engineering techniques. The system is designed to classify identified vulnerabilities as either 'malicious' or 'non-malicious', providing consensus and confidence scores to aid security analysts.

## Features

-   **Multi-Model Support:** Integrates with Causal Language Models (e.g., Phi-2, TinyLlama), Seq2Seq Language Models (e.g., Flan-T5), and Sequence Classification Models (e.g., BERT, RoBERTa).
-   **RTX 5060 Ti Optimization:** Includes optimized model loading strategies for GPUs, such as 4-bit quantization, `torch_dtype=float16`, `low_cpu_mem_usage`, and `device_map='auto'` to manage VRAM efficiently.
-   **Advanced Prompting Techniques:** Supports a range of prompt engineering strategies:
    -   **Zero-Shot:** Direct classification without examples.
    -   **Few-Shot:** Provides a small number of examples to guide the model.
    -   **Chain-of-Thought:** Encourages step-by-step reasoning for complex classifications.
    -   **Role-Based:** Assigns a persona (e.g., a cybersecurity analyst) to the model.
    -   **Hybrid:** Combines elements of different techniques for robust performance.
-   **Consensus and Confidence Scoring:** When multiple models are used, the system calculates a consensus classification and a confidence score, indicating the agreement level among models.
-   **Automated Risk Assessment:** Assigns a risk level (HIGH, MEDIUM, LOW, SAFE, UNCERTAIN) based on the consensus classification and confidence score.
-   **Manual Review Flagging:** Identifies classifications that may require manual review based on low confidence scores.
-   **Data Handling:** Supports loading vulnerabilities from CSV files and generates sample data if no input file is provided.
-   **Logging:** Comprehensive logging of processes, model loading, and errors.

## Model Types and VRAM Estimates

The system is designed to work with various Hugging Face models, with VRAM estimates for an RTX 5060 Ti (16GB VRAM):

| Model Name                                  | Type                      | VRAM Estimate |
| :------------------------------------------ | :------------------------ | :------------ |
| `microsoft/phi-2`                           | Causal-LM                 | ~3GB          |
| `TinyLlama/TinyLlama-1.1B-Chat-v1.0`        | Causal-LM                 | ~2GB          |
| `microsoft/DialoGPT-medium`                 | Causal-LM                 | ~800MB        |
| `gpt2`                                      | Causal-LM                 | ~500MB        |
| `EleutherAI/pythia-1.4b`                    | Causal-LM                 | ~3GB          |
| `stabilityai/stablelm-2-1_6b`               | Causal-LM                 | ~4GB          |
| `Qwen/Qwen2-1.5B`                           | Causal-LM                 | ~3GB          |
| `google/flan-t5-base`                       | Seq2Seq-LM                | ~1GB          |
| `google/flan-t5-large`                      | Seq2Seq-LM                | ~3GB          |
| `t5-base`                                   | Seq2Seq-LM                | ~900MB        |
| `bert-base-uncased`                         | Sequence-Classification   | ~500MB        |
| `roberta-base`                              | Sequence-Classification   | ~500MB        |
| `distilbert-base-uncased`                   | Sequence-Classification   | ~250MB        |
| `microsoft/codebert-base`                   | Sequence-Classification   | ~500MB        |
| `ehsanaghaei/SecureBERT`                    | Sequence-Classification   | ~500MB        |

## Dependencies

This project requires the following Python libraries. You can install them using `pip`:

```bash
pip install pandas torch transformers tqdm argparse enum psutil
```

Ensure you have a compatible `torch` version for your CUDA setup if using a GPU.

## Setup and Installation

1.  **Clone the repository (if applicable) or save the script:**
    ```bash
    git clone <repository-url>
    cd vulnerability-classifier
    ```
2.  **Install dependencies:**
    ```bash
    pip install -r requirements.txt # Or use the individual pip install commands above
    ```
3.  **Ensure GPU drivers are up to date (if using CUDA):** For optimal performance, ensure your NVIDIA drivers and CUDA toolkit are correctly installed and configured.

## Usage

The system can be run from the command line with various arguments:

```bash
python classifier.py --input <input_csv> --output <output_csv> --prompt-technique <technique> [--models <model1> <model2> ...] [--list-models] [--create-sample]
```

### Arguments

-   `--input`, `-i`: Path to the input CSV file containing `app_name` and `vulnerability` columns. Defaults to `vulnerabilities.csv`. If the file doesn't exist, a sample will be created.
-   `--output`, `-o`: Path to the output CSV file where results will be saved. Defaults to `results.csv`.
-   `--prompt-technique`, `-p`: Specifies the prompting strategy to use. Choices are `zero_shot`, `few_shot`, `chain_of_thought`, `role_based`, `hybrid`. Defaults to `hybrid`.
-   `--models`, `-m`: (Optional) A list of specific model names to use from the `MODEL_TYPE_MAP`. If not provided, a default set of recommended models will be used (`microsoft/phi-2`, `distilbert-base-uncased`, `google/flan-t5-base`).
-   `--list-models`, `-l`: (Optional) Prints a list of all available models and their estimated VRAM usage, then exits.
-   `--create-sample`, `-c`: (Optional) Creates a sample `vulnerabilities.csv` file with example data and exits.

### Examples

1.  **Create sample data:**
    ```bash
    python classifier.py --create-sample
    ```

2.  **Run with default models and hybrid prompting:**
    ```bash
    python classifier.py --input sample_vulnerabilities.csv --output classification_results.csv
    ```

3.  **Run with specific models and few-shot prompting:**
    ```bash
    python classifier.py -i my_vulnerabilities.csv -o my_results.csv -p few_shot -m microsoft/phi-2 google/flan-t5-base
    ```

4.  **List all compatible models:**
    ```bash
    python classifier.py --list-models
    ```

## Output Data Format

The output CSV file (`results.csv` by default) will include the original `app_name` and `vulnerability` columns, along with additional columns for each model's classification, normalized classifications, consensus, and confidence:

-   `app_name`: The name of the application.
-   `vulnerability`: The description of the vulnerability.
-   `<model_short_name>_<technique>_classification`: The raw classification output from each model (e.g., `phi_2_hybrid_classification`).
-   `<model_short_name>_<technique>_normalized`: The normalized classification (malicious, non-malicious, unknown, error) for each model.
-   `consensus_classification`: The overall classification based on the majority vote of the models (malicious, non_malicious, no_consensus).
-   `confidence_score`: A percentage score indicating the level of agreement among the models for the consensus classification (0-100%).
-   `risk_level`: Assigned risk (HIGH, MEDIUM, LOW, SAFE, UNCERTAIN) based on `consensus_classification` and `confidence_score`.
-   `requires_manual_review`: A boolean flag (`True`/`False`) indicating if the confidence score is below 70%, suggesting a need for human review.

## Code Structure

-   **`PromptTechnique` (Enum):** Defines the available prompting strategies.
-   **`MODEL_TYPE_MAP`, `MODEL_SHORT_NAMES`:** Dictionaries mapping model names to their types and short aliases.
-   **`AdvancedPromptGenerator`:** Generates prompts based on the selected `PromptTechnique`.
-   **`VulnerabilityClassifier`:** Core class for classifying vulnerabilities, handling model-specific processing (causal-lm, seq2seq-lm, sequence-classification) and response extraction.
-   **`load_model_optimized`:** Function to load models with RTX 5060 Ti specific optimizations (quantization, `torch_dtype`).
-   **`normalize_classification`:** Standardizes classification outputs to 'malicious', 'non-malicious', 'unknown', or 'error'.
-   **`calculate_consensus`, `calculate_confidence_score`:** Functions to aggregate results from multiple models.
-   **`create_sample_data`:** Generates a CSV with sample vulnerabilities for testing.
-   **`process_vulnerabilities`:** The main orchestrator function that loads data, iterates through models, classifies vulnerabilities, and saves results.
-   **`main`:** Entry point for the script, parses command-line arguments and initiates the `process_vulnerabilities` function.

In [ ]:


import pandas as pd
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification, BitsAndBytesConfig
)
import gc
import logging
from tqdm import tqdm
import json
import time
from datetime import datetime
from collections import Counter
import argparse
import os
from enum import Enum
import psutil
import warnings
warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('classifier.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class PromptTechnique(Enum):
    ZERO_SHOT = "zero_shot"
    FEW_SHOT = "few_shot"
    CHAIN_OF_THOUGHT = "chain_of_thought"
    ROLE_BASED = "role_based"
    HYBRID = "hybrid"

# RTX 5060 Ti 16GB Optimized Model Selection
MODEL_TYPE_MAP = {
    # CAUSAL LM MODELS (RTX 5060 Ti Compatible)
    "microsoft/phi-2": "causal-lm",                    # ~3GB VRAM
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0": "causal-lm", # ~2GB VRAM
    "microsoft/DialoGPT-medium": "causal-lm",          # ~800MB VRAM
    "gpt2": "causal-lm",                               # ~500MB VRAM
    "EleutherAI/pythia-1.4b": "causal-lm",            # ~3GB VRAM
    "stabilityai/stablelm-2-1_6b": "causal-lm",       # ~4GB VRAM
    "Qwen/Qwen2-1.5B": "causal-lm",                   # ~3GB VRAM

    # SEQ2SEQ LM MODELS
    "google/flan-t5-base": "seq2seq-lm",              # ~1GB VRAM
    "google/flan-t5-large": "seq2seq-lm",             # ~3GB VRAM
    "t5-base": "seq2seq-lm",                          # ~900MB VRAM

    # SEQUENCE CLASSIFICATION MODELS
    "bert-base-uncased": "sequence-classification",    # ~500MB VRAM
    "roberta-base": "sequence-classification",         # ~500MB VRAM
    "distilbert-base-uncased": "sequence-classification", # ~250MB VRAM
    "microsoft/codebert-base": "sequence-classification", # ~500MB VRAM
    "ehsanaghaei/SecureBERT": "sequence-classification", # ~500MB VRAM
}

MODEL_SHORT_NAMES = {k: k.split("/")[-1].replace("-", "_").replace(".", "_") for k in MODEL_TYPE_MAP}

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"VRAM: {gpu_memory:.1f}GB")

class AdvancedPromptGenerator:
    """Advanced prompt generator for multiple prompting techniques"""

    def __init__(self):
        self.few_shot_examples = [
            {
                "app": "WebApp",
                "vulnerability": "SQL injection in login allows authentication bypass",
                "classification": "malicious"
            },
            {
                "app": "E-commerce",
                "vulnerability": "XSS in comments enables session hijacking",
                "classification": "malicious"
            },
            {
                "app": "Portal",
                "vulnerability": "Missing security headers - informational finding",
                "classification": "non-malicious"
            },
            {
                "app": "API",
                "vulnerability": "Outdated library with no known exploits",
                "classification": "non-malicious"
            }
        ]

    def generate_zero_shot_prompt(self, app_name: str, vulnerability: str) -> str:
        """Zero-shot prompting"""
        return (
            f"Classify this vulnerability as 'malicious' or 'non-malicious':\n\n"
            f"Application: {app_name}\n"
            f"Vulnerability: {vulnerability}\n\n"
            f"Classification:"
        )

    def generate_few_shot_prompt(self, app_name: str, vulnerability: str) -> str:
        """Few-shot prompting with examples"""
        examples = "\n".join([
            f"App: {ex['app']} | Issue: {ex['vulnerability']} | Result: {ex['classification']}"
            for ex in self.few_shot_examples[:2]  # Use 2 examples for efficiency
        ])

        return (
            f"Learn from examples:\n{examples}\n\n"
            f"Now classify:\nApp: {app_name} | Issue: {vulnerability}\n\n"
            f"Result:"
        )

    def generate_chain_of_thought_prompt(self, app_name: str, vulnerability: str) -> str:
        """Chain-of-thought reasoning"""
        return (
            f"Analyze this vulnerability step-by-step:\n\n"
            f"App: {app_name}\n"
            f"Issue: {vulnerability}\n\n"
            f"Analysis:\n"
            f"1. Exploitability: Can this be exploited remotely?\n"
            f"2. Impact: What damage could occur?\n"
            f"3. Risk Level: High or Low?\n"
            f"4. Classification: malicious or non-malicious\n\n"
            f"Step 1:"
        )

    def generate_role_based_prompt(self, app_name: str, vulnerability: str) -> str:
        """Role-based prompting"""
        return (
            f"You are a senior cybersecurity analyst with 10+ years experience.\n\n"
            f"Application: {app_name}\n"
            f"Finding: {vulnerability}\n\n"
            f"Based on your expertise, classify this as:\n"
            f"- 'malicious': Significant security risk\n"
            f"- 'non-malicious': Low/no security risk\n\n"
            f"Professional assessment:"
        )

    def generate_hybrid_prompt(self, app_name: str, vulnerability: str) -> str:
        """Hybrid approach combining techniques"""
        return (
            f"SECURITY ANALYSIS\n"
            f"App: {app_name}\n"
            f"Finding: {vulnerability}\n\n"
            f"RULES:\n"
            f"- malicious: Exploitable, data breach, system compromise\n"
            f"- non-malicious: Informational, low impact, theoretical\n\n"
            f"EXAMPLES:\n"
            f"- SQL injection -> malicious\n"
            f"- Missing headers -> non-malicious\n\n"
            f"Classification:"
        )

class VulnerabilityClassifier:
    """Enhanced classifier with multiple prompting strategies"""

    def __init__(self, prompt_technique: PromptTechnique = PromptTechnique.HYBRID):
        self.prompt_generator = AdvancedPromptGenerator()
        self.prompt_technique = prompt_technique

    def classify_vulnerability(self, app_name: str, vulnerability: str, tokenizer, model,
                             model_type: str, model_name: str = None, max_retries: int = 2):
        """Enhanced classification with optimized generation"""

        for attempt in range(max_retries):
            try:
                prompt = self._generate_prompt(app_name, vulnerability)

                if model_type == "causal-lm":
                    return self._process_causal_lm(prompt, tokenizer, model, model_name)
                elif model_type == "seq2seq-lm":
                    return self._process_seq2seq_lm(prompt, tokenizer, model)
                elif model_type == "sequence-classification":
                    return self._process_classification_model(app_name, vulnerability, tokenizer, model)
                else:
                    return "unsupported-model-type"

            except Exception as e:
                logger.warning(f"Attempt {attempt + 1} failed: {str(e)}")
                if attempt == max_retries - 1:
                    return f"Error: {str(e)[:50]}"
                time.sleep(0.5)

    def _generate_prompt(self, app_name: str, vulnerability: str) -> str:
        """Generate prompt based on technique"""
        if self.prompt_technique == PromptTechnique.ZERO_SHOT:
            return self.prompt_generator.generate_zero_shot_prompt(app_name, vulnerability)
        elif self.prompt_technique == PromptTechnique.FEW_SHOT:
            return self.prompt_generator.generate_few_shot_prompt(app_name, vulnerability)
        elif self.prompt_technique == PromptTechnique.CHAIN_OF_THOUGHT:
            return self.prompt_generator.generate_chain_of_thought_prompt(app_name, vulnerability)
        elif self.prompt_technique == PromptTechnique.ROLE_BASED:
            return self.prompt_generator.generate_role_based_prompt(app_name, vulnerability)
        else:  # HYBRID
            return self.prompt_generator.generate_hybrid_prompt(app_name, vulnerability)

    def _process_causal_lm(self, prompt: str, tokenizer, model, model_name: str):
        """Process causal LM models with RTX 5060 Ti optimizations"""
        max_tokens = 25 if self.prompt_technique == PromptTechnique.CHAIN_OF_THOUGHT else 10

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True,
            return_attention_mask=True
        ).to(device)

        # RTX 5060 Ti optimized parameters
        generation_params = {
            "input_ids": inputs['input_ids'],
            "attention_mask": inputs['attention_mask'],
            "max_new_tokens": max_tokens,
            "min_new_tokens": 1,
            "pad_token_id": tokenizer.pad_token_id,
            "do_sample": True,
            "temperature": 0.7,
            "top_p": 0.9,
            "repetition_penalty": 1.1,
            "use_cache": True,
        }

        # Model-specific tuning
        if model_name and "phi" in model_name.lower():
            generation_params.update({"temperature": 0.6, "top_p": 0.95})
        elif model_name and "llama" in model_name.lower():
            generation_params.update({"do_sample": False, "num_beams": 2})

        with torch.no_grad():
            outputs = model.generate(**generation_params)

        response = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        ).strip()

        return self._extract_classification(response)

    def _process_seq2seq_lm(self, prompt: str, tokenizer, model):
        """Process seq2seq models"""
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                num_beams=1,
                use_cache=True
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return self._extract_classification(response)

    def _process_classification_model(self, app_name: str, vulnerability: str, tokenizer, model):
        """Process classification models"""
        text = f"App: {app_name} | Vulnerability: {vulnerability}"
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            predicted_class_id = outputs.logits.argmax().item()

            if hasattr(model.config, "id2label"):
                label = model.config.id2label.get(predicted_class_id, str(predicted_class_id))
                return label.lower()
            else:
                return "malicious" if predicted_class_id == 1 else "non-malicious"

    def _extract_classification(self, response: str) -> str:
        """Extract classification from response"""
        if not response:
            return "unknown"

        response_lower = response.lower().strip()

        # Chain-of-thought special handling
        if self.prompt_technique == PromptTechnique.CHAIN_OF_THOUGHT:
            lines = response.split('\n')
            for line in reversed(lines):
                if 'malicious' in line.lower():
                    return 'malicious' if 'non-malicious' not in line.lower() else 'non-malicious'
                elif 'non-malicious' in line.lower():
                    return 'non-malicious'

        # Standard extraction
        if 'malicious' in response_lower and 'non-malicious' not in response_lower:
            return "malicious"
        elif 'non-malicious' in response_lower or 'safe' in response_lower:
            return "non-malicious"
        elif any(word in response_lower for word in ['dangerous', 'critical', 'vulnerable']):
            return "malicious"
        elif any(word in response_lower for word in ['minor', 'informational', 'low']):
            return "non-malicious"

        return "unknown"

def load_model_optimized(model_name, model_type, use_quantization=True):
    """RTX 5060 Ti optimized model loading"""
    try:
        logger.info(f"Loading {model_name}...")

        load_kwargs = {
            "trust_remote_code": True,
            "torch_dtype": torch.float16,
            "low_cpu_mem_usage": True,
            "device_map": "auto"
        }

        # Quantization for large models
        if use_quantization and ("7b" in model_name.lower() or "large" in model_name.lower()):
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4"
            )
            load_kwargs["quantization_config"] = quantization_config
            logger.info("Using 4-bit quantization")

        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

        if model_type == "causal-lm":
            model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
        elif model_type == "seq2seq-lm":
            model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **load_kwargs)
        elif model_type == "sequence-classification":
            model = AutoModelForSequenceClassification.from_pretrained(model_name, **load_kwargs)
        else:
            raise ValueError(f"Unknown model_type: {model_type}")

        # Handle tokenizer padding
        if tokenizer.pad_token is None:
            if tokenizer.eos_token:
                tokenizer.pad_token = tokenizer.eos_token
            else:
                tokenizer.add_special_tokens({'pad_token': '[PAD]'})
                model.resize_token_embeddings(len(tokenizer))

        model.eval()

        # Memory info
        if device == "cuda":
            memory_used = torch.cuda.memory_allocated() / (1024**3)
            logger.info(f"Model loaded | VRAM: {memory_used:.1f}GB")

        return tokenizer, model

    except Exception as e:
        logger.error(f"Failed to load {model_name}: {e}")
        raise

def normalize_classification(text):
    """Normalize classification responses"""
    if pd.isna(text) or text == "":
        return "unknown"

    text_lower = str(text).lower().strip()

    if "error" in text_lower:
        return "error"
    elif any(word in text_lower for word in ['malicious', 'dangerous', 'critical', 'vulnerable']):
        return "malicious"
    elif any(word in text_lower for word in ['non-malicious', 'safe', 'secure', 'low', 'informational']):
        return "non-malicious"
    else:
        return "unknown"

def calculate_consensus(row, classification_cols):
    """Calculate consensus from multiple models"""
    classifications = [normalize_classification(row[col]) for col in classification_cols]
    valid_classifications = [c for c in classifications if c not in ["error", "unknown"]]

    if not valid_classifications:
        return "no_consensus"

    votes = Counter(valid_classifications)
    return votes.most_common(1)[0][0] if votes else "no_consensus"

def calculate_confidence_score(row, classification_cols):
    """Calculate confidence score (0-100%)"""
    classifications = [normalize_classification(row[col]) for col in classification_cols]
    valid_classifications = [c for c in classifications if c not in ["error", "unknown"]]

    if not valid_classifications:
        return 0.0

    votes = Counter(valid_classifications)
    most_common_count = votes.most_common(1)[0][1]
    confidence = (most_common_count / len(valid_classifications)) * 100

    return round(confidence, 2)

def create_sample_data(filename="sample_vulnerabilities.csv"):
    """Create sample vulnerability data"""
    sample_data = {
        'app_name': [
            'WebApp Authentication',
            'E-commerce Platform',
            'Corporate Portal',
            'API Gateway',
            'Mobile Banking App',
            'Content Management System',
            'Database Server',
            'File Upload Service'
        ],
        'vulnerability': [
            'SQL injection vulnerability in login form allows authentication bypass',
            'Cross-site scripting (XSS) in product reviews enables session hijacking',
            'Missing HTTP security headers - informational finding only',
            'Buffer overflow in native library enables remote code execution',
            'Outdated SSL/TLS configuration with weak cipher suites',
            'Directory traversal vulnerability allows file system access',
            'Insecure direct object references expose user data',
            'Missing input validation in file upload allows malicious file execution'
        ]
    }

    df = pd.DataFrame("dset.csv")
    df.to_csv(filename, index=False)
    logger.info(f"Sample data created: {filename}")
    return df

def process_vulnerabilities(input_file, output_file, prompt_technique=PromptTechnique.HYBRID,
                          models_filter=None, save_frequency=5):
    """Main processing function"""
    start_time = time.time()

    # Create sample data if input doesn't exist
    if not os.path.exists(input_file):
        logger.info("Input file not found, creating sample data...")
        df = create_sample_data(input_file)
    else:
        df = pd.read_csv(input_file)

    logger.info(f"Processing {len(df)} vulnerabilities")
    logger.info(f"Technique: {prompt_technique.value}")

    # Initialize classifier
    classifier = VulnerabilityClassifier(prompt_technique)

    # Select models
    if models_filter is None:
        # Default RTX 5060 Ti recommended models
        recommended_models = [
            "microsoft/phi-2",
            "distilbert-base-uncased",
            "google/flan-t5-base"
        ]
        models_to_use = {k: v for k, v in MODEL_TYPE_MAP.items() if k in recommended_models}
    else:
        models_to_use = {k: v for k, v in MODEL_TYPE_MAP.items() if k in models_filter}

    logger.info(f"Using {len(models_to_use)} models: {list(models_to_use.keys())}")

    classification_cols = []

    for model_name, model_type in models_to_use.items():
        col = f"{MODEL_SHORT_NAMES[model_name]}_{prompt_technique.value}_classification"
        classification_cols.append(col)
        df[col] = ""

        logger.info(f"\nProcessing: {model_name}")

        try:
            # Load model
            tokenizer, model = load_model_optimized(model_name, model_type)

            # Process each vulnerability
            for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Classifying"):
                app_name = str(row['app_name'])
                vulnerability = str(row['vulnerability'])

                result = classifier.classify_vulnerability(
                    app_name, vulnerability, tokenizer, model, model_type, model_name
                )

                df.at[idx, col] = result

                # Save progress
                if (idx + 1) % save_frequency == 0:
                    df.to_csv(output_file, index=False)

            # Cleanup
            del model, tokenizer
            gc.collect()
            torch.cuda.empty_cache()

            memory_used = torch.cuda.memory_allocated() / (1024**3) if device == "cuda" else 0
            logger.info(f"Memory cleaned | VRAM: {memory_used:.1f}GB")

        except Exception as e:
            logger.error(f"Error with {model_name}: {e}")
            df[col] = f"Error: {str(e)[:30]}"

    # Generate results
    logger.info("\nGenerating final results...")

    # Normalize classifications
    for col in classification_cols:
        normalized_col = col.replace("_classification", "_normalized")
        df[normalized_col] = df[col].apply(normalize_classification)

    # Calculate consensus and confidence
    if len(classification_cols) > 1:
        df['consensus_classification'] = df.apply(
            lambda row: calculate_consensus(row, classification_cols), axis=1
        )
        df['confidence_score'] = df.apply(
            lambda row: calculate_confidence_score(row, classification_cols), axis=1
        )

        # Risk assessment
        def assign_risk_level(row):
            if row['consensus_classification'] == 'malicious':
                if row['confidence_score'] >= 80:
                    return "HIGH"
                elif row['confidence_score'] >= 60:
                    return "MEDIUM"
                else:
                    return "LOW"
            elif row['consensus_classification'] == 'non-malicious':
                return "SAFE"
            else:
                return "UNCERTAIN"

        df['risk_level'] = df.apply(assign_risk_level, axis=1)
        df['requires_manual_review'] = df['confidence_score'] < 70

    # Save final results
    df.to_csv(output_file, index=False)

    processing_time = time.time() - start_time
    logger.info(f"\nProcessing completed in {processing_time:.2f} seconds")
    logger.info(f"Results saved to: {output_file}")

    # Summary
    logger.info(f"\nRESULTS SUMMARY:")
    for col in classification_cols:
        results = df[col].apply(normalize_classification).value_counts()
        logger.info(f"  {MODEL_SHORT_NAMES[model_name]}:")
        for classification, count in results.items():
            logger.info(f"    {classification}: {count}")

    if 'consensus_classification' in df.columns:
        logger.info(f"\nCONSENSUS RESULTS:")
        consensus_results = df['consensus_classification'].value_counts()
        for classification, count in consensus_results.items():
            logger.info(f"  {classification}: {count}")

        logger.info(f"\nRISK LEVELS:")
        risk_results = df['risk_level'].value_counts()
        for risk, count in risk_results.items():
            logger.info(f"  {risk}: {count}")

    return df

def main():
    parser = argparse.ArgumentParser(description='RTX 5060 Ti Vulnerability Classification System')
    parser.add_argument('--input', '-i', default='vulnerabilities.csv', help='Input CSV file')
    parser.add_argument('--output', '-o', default='results.csv', help='Output CSV file')
    parser.add_argument('--prompt-technique', '-p',
                       choices=[t.value for t in PromptTechnique],
                       default=PromptTechnique.HYBRID.value,
                       help='Prompting technique')
    parser.add_argument('--models', '-m', nargs='+', help='Specific models to use')
    parser.add_argument('--list-models', '-l', action='store_true', help='List available models')
    parser.add_argument('--create-sample', '-c', action='store_true', help='Create sample data and exit')

    args = parser.parse_args()

    if args.list_models:
        logger.info("\nRTX 5060 Ti Compatible Models:")
        logger.info("=" * 50)
        for model_name, model_type in MODEL_TYPE_MAP.items():
            vram_est = "~1GB" if any(x in model_name.lower() for x in ["distil", "bert"]) else \
                      "~2-3GB" if any(x in model_name.lower() for x in ["phi", "tiny"]) else "~3-5GB"
            logger.info(f"{model_name} ({model_type}) - {vram_est}")
        return

    if args.create_sample:
        create_sample_data(args.input)
        logger.info(f"Sample data created: {args.input}")
        return

    # Run processing
    technique = PromptTechnique(args.prompt_technique)
    result_df = process_vulnerabilities(
        input_file=args.input,
        output_file=args.output,
        prompt_technique=technique,
        models_filter=args.models
    )

    logger.info(f"\nClassification complete! Check {args.output} for results.")

if __name__ == "__main__":
    main()
